# 26 - Multi-organism biocide dataset readiness audit

This notebook asks a narrow question:

Can any organism beyond *Listeria monocytogenes* support a second supervised biocide-tolerance benchmark using only public data?

Strict inclusion rule:

- row-level biocide phenotype labels are required;
- isolate IDs must map to public genome/SRA/assembly accessions;
- aggregate MIC distributions, figure-only data, and experimental-evolution labels are not enough for training;
- small datasets may still be useful as external case studies.


In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

AUDIT_CSV = PROJECT_ROOT / "data" / "manifests" / "multiorganism_biocide_ml_readiness_audit.csv"
REPORT = PROJECT_ROOT / "reports" / "multiorganism_biocide_ml_readiness_audit.json"


## 1. Run the audit

In [ ]:
command = [sys.executable, str(PROJECT_ROOT / "scripts" / "audit_multiorganism_ml_ready_biocide_datasets.py")]
print(" ".join(command))
subprocess.run(command, cwd=PROJECT_ROOT, check=True)
report = json.loads(REPORT.read_text())
report["summary"]


## 2. Ranked candidates

In [ ]:
audit = pd.read_csv(AUDIT_CSV)
display(audit[[
    "priority", "study_key", "organism_group", "endpoint", "reported_n_phenotyped",
    "reported_n_sequenced", "row_level_labels_status", "public_genome_status",
    "accession_or_project", "ml_readiness", "audit_batch", "doi"
]])


## 3. Immediate audit batch

In [ ]:
immediate = audit[audit["audit_batch"].eq("immediate")].copy()
display(immediate[[
    "study_key", "organism_group", "endpoint", "row_level_label_evidence",
    "genome_linkage_evidence", "next_action", "caution", "url"
]])


## 4. Small external-case candidates

In [ ]:
external = audit[audit["audit_batch"].eq("external_case")].copy()
display(external[[
    "study_key", "organism_group", "endpoint", "reported_n_phenotyped",
    "accession_or_project", "next_action", "caution", "url"
]])


## 5. Context-only or mechanistic datasets

In [ ]:
context = audit[audit["audit_batch"].eq("context")].copy()
display(context[[
    "study_key", "organism_group", "endpoint", "ml_readiness",
    "row_level_labels_status", "public_genome_status", "caution", "url"
]])


## 6. Decision

The next bounded batch is:

1. *Proteus mirabilis* 2025 cationic-biocide dataset.
2. *Enterococcus faecium* 2019 chlorhexidine dataset.
3. *Enterococcus faecalis* 2022 chlorhexidine dataset.
4. Hartmann lab mixed hospital-environment chlorhexidine preprint.

Do not train yet. The next notebook should extract or verify supplements for this batch and produce one of two outcomes:

- a row-level phenotype-genome manifest for a second supervised benchmark; or
- a documented stop decision that no public non-*Listeria* organism is ML-ready without author contact or manual data recovery.

The strongest current paper remains the *Listeria* supervised benchmark plus a transparent, evidence-based audit showing why pharma-relevant organisms are exploratory rather than validated tolerance predictors.
